# 02 - Elo Baseline

**Goal:** a rating for every team before every game (2012–2025), and a pre-game
win probability for each game.

**Why it matters:** Elo is the backbone of the project. Every later model uses the
rating gap as its main input, and every later model has to beat Elo to justify
its complexity. It is also the first version that can run live in 2026.

**Requires:** `data/games_clean.parquet`. Run `01_data_exploration.ipynb` first.

**Approach:** build the simplest version first, then add one feature at a time and
measure whether each one helps:
1. Basic Elo: every team starts at 1500, fixed K
2. + home-field advantage (none at neutral sites)
3. + margin-of-victory multiplier
4. + offseason regression toward 1500
5. Tune K and HFA on 2012–2021 only, by Brier score

**Rule:** predictions are saved *before* each game's update. No look-ahead.

## Contents
1. Setup and load
2. One Elo update by hand
3. Elo building blocks (functions)
4. The core loop
5. Adding features
6. Scoring: Brier and accuracy
7. Tuning K and HFA
8. Rating histories (sanity check)
9. Findings and decisions

## Setup and load

In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt

games = pd.read_parquet("../data/games_clean.parquet")
games.shape

(3829, 21)

In [2]:
print(games.iloc[0])

game_id               2012_01_DAL_NYG
season                           2012
week                                1
kickoff           2012-09-05 20:30:00
game_type                         REG
is_playoff                      False
home_team                         NYG
away_team                         DAL
location                         Home
is_neutral                      False
home_score                         17
away_score                         24
result                             -7
home_win                          0.0
home_rest                           7
away_rest                           7
home_qb_id                 00-0022803
away_qb_id                 00-0021678
spread_line                       3.5
home_moneyline                 -182.0
away_moneyline                  164.0
Name: 0, dtype: object


## 3. Elo Building Blocks

In [3]:
def expected_score(rating_home, rating_away, hfa=0):
    """Home team's expected result (0 to 1) before the game. """
    d = rating_home - rating_away + hfa
    return 1 / (1 + 10 ** (-d / 400))

In [4]:
def mov_multiplier(margin, elo_diff_winner):
    """
    Margin of victory multiplier (FiveThirtyEight's form)
    margin: home points minus away points
    elo_diff_winner: winner's rating minus loser's, including HFA on the home side
    """
    if margin == 0:
        return 1.0
    return np.log(abs(margin) + 1) * 2.2 / (0.001 * elo_diff_winner + 2.2)

In [5]:
def regress_ratings(ratings, fraction, mean=1500):
    """ Pull every rating part of the way back toward the mean in the offseason"""
    return {team: r + fraction * (mean - r) for team, r in ratings.items()}

In [6]:
# Test
e = expected_score(1500, 1500, hfa=48)
m = mov_multiplier(margin=-7, elo_diff_winner=1500 - (1500 + 48))
print(f"E_home = {e:.3f}, MOV mult = {m:.2f}, change = {20 * m * (0 - e):.1f}")

E_home = 0.569, MOV mult = 2.13, change = -24.2
